# 00 — Smoke test
Run after `--limit 1` invocations of the sweep script. Shows input face next to one Stage 2 and (if available) one Stage 3 generation.

In [ ]:
import sys, os
from pathlib import Path
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO))
os.chdir(REPO)

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

In [ ]:
def latest_run(stage: int):
    runs = sorted(Path('experiments/runs').glob(f'stage{stage}_*'))
    return runs[-1] if runs else None

stage2 = latest_run(2); stage3 = latest_run(3)
print('stage2:', stage2)
print('stage3:', stage3)

In [ ]:
def first_image(run_dir):
    if run_dir is None: return None, None
    df = pd.read_parquet(run_dir / 'metadata.parquet')
    df = df[df['image_path'] != '']
    if df.empty: return None, None
    row = df.iloc[0]
    return Image.open(row['face_path']), Image.open(row['image_path'])

face2, out2 = first_image(stage2)
face3, out3 = first_image(stage3)

panels = []
if face2 is not None: panels.append(('Stage 2 input', face2)); panels.append(('Stage 2 output', out2))
if face3 is not None: panels.append(('Stage 3 input', face3)); panels.append(('Stage 3 output', out3))

fig, axes = plt.subplots(1, len(panels), figsize=(4*len(panels), 4))
if len(panels) == 1: axes = [axes]
for ax, (title, img) in zip(axes, panels):
    ax.imshow(img); ax.set_title(title); ax.axis('off')
plt.tight_layout(); plt.show()